# ¿Es asequible el precio de la vivienda para el cubano promedio?
## Análisis de anuncios inmobiliarios de La Habana

**Proyecto académico - Ciencia de Datos (ICD 2026)**
Caso de estudio: La Habana, Cuba

Este notebook tiene como objetivo analizar si el precio de la vivienda en La Habana es asequible para el cubano promedio, a partir de anuncios publicados en portales inmobiliarios (GAOS y Revolico).

El análisis busca responder a las siguientes preguntas:
1. ¿Cuánto cuesta una casa en La Habana?
2. ¿Cuánto cuesta un apartamento en La Habana?
3. ¿Cómo se distribuyen los precios de las viviendas entre los municipios de La Habana?
4. ¿Qué características aparecen con mayor frecuencia en los anuncios cuyo precio supera la mediana?
5. ¿Cómo se relacionan los precios de las viviendas con los salarios establecidos oficialmente?
6. ¿Cuál es el inmueble más habitual y cuánto cuesta en relación con el salario mínimo?

**Nota metodológica:** En este proyecto no se calcula el precio por metro cuadrado, ya que la información de áreas es incompleta y sus unidades no están verificadas. Tampoco se intenta determinar qué características hacen que un precio sea superior o inferior a la mediana, ya que los datos disponibles no permiten establecer esa relación de manera fiable.

**En esta primera etapa:** Cargamos los datos y realizamos una primera exploración para comprender qué información tenemos disponible.

## 2. Importaciones

Importamos las librerías necesarias para esta primera fase de exploración de datos.

In [ ]:
import pandas as pd
import json

## 3. Carga de datos

Cargamos el archivo unificado que contiene los anuncios de GAOS y Revolico. El archivo tiene una estructura anidada: los anuncios se encuentran bajo la clave `'datos'`, mientras que el nivel superior contiene metadatos de la captura (fecha, fuente, etc.).

In [ ]:
ruta_archivo = 'data/processed/anuncios/unificado_procesado_83924fda_20261115_034632.json'

with open(ruta_archivo, 'r', encoding='utf-8') as f:
    datos = json.load(f)

anuncios = datos['datos']
df = pd.DataFrame(anuncios)

print(f'Anuncios cargados: {len(df)}')
print(f'Columnas: {df.columns.tolist()}')

## 4. Primera exploración de datos

En esta sección realizamos una primera revisión de los datos cargados para entender su estructura y calidad.

In [ ]:
# Dimensiones del DataFrame
print('Dimensiones del DataFrame:')
print(df.shape)
print(f'Filas (anuncios): {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

In [ ]:
# Primeros registros
print('Primeros 5 anuncios:')
df.head()

In [ ]:
# Nombres de las columnas
print('Columnas del DataFrame:')
for columna in df.columns:
    print(f'  - {columna}')

In [ ]:
# Tipos de datos de cada columna
print('Tipos de datos:')
print(df.dtypes)

In [ ]:
# Información general del DataFrame
df.info()

In [ ]:
# Cantidad y porcentaje de valores nulos por columna
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({'nulos': nulos, 'porcentaje': porcentaje_nulos})
tabla_nulos = tabla_nulos[tabla_nulos['nulos'] > 0].sort_values('nulos', ascending=False)

print('Columnas con valores nulos:')
print(tabla_nulos)

In [ ]:
# Cantidad de anuncios por fuente
print('Anuncios por fuente:')
print(df['fuente'].value_counts())

print('\nPorcentaje por fuente:')
print((df['fuente'].value_counts() / len(df) * 100).round(2))

## 5. Exploración de variables clave

Antes de analizar los precios, revisamos cómo están distribuidas las variables que utilizaremos en el análisis: tipo de operación, tipo de inmueble, moneda, provincia, municipio, número de habitaciones, número de baños y área en metros cuadrados.

In [ ]:
# Distribución de cada variable clave
variables = ['tipo_operacion', 'tipo_inmueble', 'moneda', 'provincia', 'municipio', 'habitaciones', 'banos', 'area_m2']

for variable in variables:
    print(f'--- {variable} ---')
    print(df[variable].value_counts(dropna=False))
    print()

## 6. Comprobaciones específicas

Verificamos aspectos importantes antes de comenzar el análisis de precios: si todos los anuncios son de venta, qué tipos de inmueble aparecen, qué monedas se utilizan y cuántos anuncios tienen precio y características registradas.

In [ ]:
# 1. ¿Todos los anuncios son de venta?
print('1. ¿Todos los anuncios son de venta?')
print(df['tipo_operacion'].value_counts(dropna=False))

# 2. ¿Qué categorías aparecen en tipo_inmueble?
print('\n2. Categorías de tipo_inmueble:')
print(df['tipo_inmueble'].value_counts(dropna=False))

# 3. ¿Cuántos anuncios tienen precio en USD, EUR, otras monedas o sin moneda?
print('\n3. Monedas:')
print(df['moneda'].value_counts(dropna=False))

# 4. ¿Cuántos anuncios tienen precio registrado y cuántos no, diferenciando por moneda?
print('\n4. Precio registrado por moneda:')
con_precio = df[df['precio'].notna()]
print(f'Anuncios con precio registrado: {len(con_precio)}')
print(con_precio['moneda'].value_counts(dropna=False))
print(f'\nAnuncios sin precio registrado: {len(df) - len(con_precio)}')
print(df[df['precio'].isna()]['moneda'].value_counts(dropna=False))

# 5. ¿Cuántos anuncios tienen información sobre habitaciones, baños y área?
print('\n5. Información sobre características:')
print(f'Con habitaciones: {df["habitaciones"].notna().sum()}')
print(f'Con baños: {df["banos"].notna().sum()}')
print(f'Con área (m²): {df["area_m2"].notna().sum()}')

## 7. Precio de las casas en La Habana (USD)

En esta sección respondemos a la primera pregunta del proyecto: ¿cuánto cuesta una casa en La Habana? Para ello, filtramos los anuncios que cumplen tres condiciones: sean clasificados como `casa`, tengan precio registrado y estén expresados en dólares estadounidenses (USD). No convertimos monedas ni mezclamos con EUR, y no eliminamos precios extremos para conservar la información original.

In [ ]:
# Filtrar anuncios: tipo_inmueble == 'casa', precio no nulo y moneda == 'USD'
df_casas_usd = df[
    (df["tipo_inmueble"] == "casa") &
    (df["precio"].notna()) &
    (df["moneda"] == "USD")
].copy()

print(f"Anuncios totales: {len(df)}")
print(f"Anuncios de casas en USD: {len(df_casas_usd)}")
print(f"Porcentaje respecto al total: {len(df_casas_usd) / len(df) * 100:.2f}%")

In [ ]:
# Estadísticas descriptivas del precio
precios = df_casas_usd["precio"]

print(f"Cantidad de anuncios: {len(precios)}")
print(f"Media: {precios.mean():,.2f} USD")
print(f"Mediana: {precios.median():,.2f} USD")
print(f"Mínimo: {precios.min():,.2f} USD")
print(f"Máximo: {precios.max():,.2f} USD")
print(f"Primer cuartil (Q1, 25%): {precios.quantile(0.25):,.2f} USD")
print(f"Tercer cuartil (Q3, 75%): {precios.quantile(0.75):,.2f} USD")

In [ ]:
# Tabla resumen de estadísticas
tabla_estadisticas = pd.DataFrame({
    "estadístico": ["Cantidad", "Media", "Mediana", "Mínimo", "Máximo", "Q1 (25%)", "Q3 (75%)"],
    "valor (USD)": [
        len(precios),
        f"{precios.mean():,.2f}",
        f"{precios.median():,.2f}",
        f"{precios.min():,.2f}",
        f"{precios.max():,.2f}",
        f"{precios.quantile(0.25):,.2f}",
        f"{precios.quantile(0.75):,.2f}",
    ],
})

print(tabla_estadisticas.to_string(index=False))

### Histograma de precios

El histograma muestra cuántos anuncios hay en cada rango de precios, permitiendo observar la distribución y detectar concentraciones de precios.

In [ ]:
# Histograma de precios
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(precios, bins=20, edgecolor="black", color="steelblue")
plt.title("Distribución de precios de casas en USD")
plt.xlabel("Precio (USD)")
plt.ylabel("Número de anuncios")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Diagrama de caja (boxplot)

El boxplot muestra la dispersión de los precios, la mediana, los cuartiles y los posibles valores atípicos individuales.

In [ ]:
# Diagrama de caja (boxplot)
plt.figure(figsize=(6, 5))
plt.boxplot(
    precios,
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", edgecolor="black"),
    medianprops=dict(color="red", linewidth=2),
)
plt.title("Boxplot de precios de casas en USD")
plt.ylabel("Precio (USD)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Análisis de precios extremos

En esta sección revisamos los anuncios con el precio mínimo y el precio máximo entre las casas en USD. El objetivo es entender si esos valores extremos tienen una explicación en los datos disponibles, sin eliminar ni modificar ningún anuncio.

In [ ]:
# Identificar precios mínimo y máximo
precio_min = precios.min()
precio_max = precios.max()

print(f"Precio mínimo: {precio_min:,.2f} USD")
print(f"Precio máximo: {precio_max:,.2f} USD")
print()

# Anuncios con precio mínimo
anuncios_min = df_casas_usd[df_casas_usd["precio"] == precio_min]
print(f"Anuncios con precio mínimo ({precio_min:,.2f} USD): {len(anuncios_min)}")
print("=" * 80)

columnas_mostrar = [
    "id_fuente", "fuente", "titulo", "descripcion", "precio", "moneda",
    "tipo_inmueble", "municipio", "direccion_texto", "area_m2",
    "habitaciones", "banos", "url",
]

for _, anuncio in anuncios_min.iterrows():
    for col in columnas_mostrar:
        valor = anuncio.get(col)
        if pd.isna(valor):
            valor = "(no disponible)"
        print(f"{col}: {valor}")
    print("-" * 80)

# Anuncios con precio máximo
anuncios_max = df_casas_usd[df_casas_usd["precio"] == precio_max]
print(f"\nAnuncios con precio máximo ({precio_max:,.2f} USD): {len(anuncios_max)}")
print("=" * 80)

for _, anuncio in anuncios_max.iterrows():
    for col in columnas_mostrar:
        valor = anuncio.get(col)
        if pd.isna(valor):
            valor = "(no disponible)"
        print(f"{col}: {valor}")
    print("-" * 80)

### Observaciones

Revisa si el título y la descripción de cada anuncio permiten entender por qué el precio es tan bajo o tan alto. Los datos faltantes (campos vacíos) también son relevantes para interpretar los precios extremos.

## 9. Precio de los apartamentos en La Habana (USD)

En esta sección respondemos a la segunda pregunta del proyecto: ¿cuánto cuesta un apartamento en La Habana? Para ello, filtramos los anuncios que cumplen tres condiciones: sean clasificados como `apartamento`, tengan precio registrado y estén expresados en dólares estadounidenses (USD). No convertimos monedas ni mezclamos con EUR, y no eliminamos precios extremos para conservar la información original.

In [ ]:
# Filtrar anuncios: tipo_inmueble == 'apartamento', precio no nulo y moneda == 'USD'
df_apartamentos_usd = df[
    (df["tipo_inmueble"] == "apartamento") &
    (df["precio"].notna()) &
    (df["moneda"] == "USD")
].copy()

print(f"Anuncios totales: {len(df)}")
print(f"Anuncios de apartamentos en USD: {len(df_apartamentos_usd)}")
print(f"Porcentaje respecto al total: {len(df_apartamentos_usd) / len(df) * 100:.2f}%")

In [ ]:
# Estadísticas descriptivas del precio
precios_ap = df_apartamentos_usd["precio"]

print(f"Cantidad de anuncios: {len(precios_ap)}")
print(f"Media: {precios_ap.mean():,.2f} USD")
print(f"Mediana: {precios_ap.median():,.2f} USD")
print(f"Mínimo: {precios_ap.min():,.2f} USD")
print(f"Máximo: {precios_ap.max():,.2f} USD")
print(f"Primer cuartil (Q1, 25%): {precios_ap.quantile(0.25):,.2f} USD")
print(f"Tercer cuartil (Q3, 75%): {precios_ap.quantile(0.75):,.2f} USD")

In [ ]:
# Tabla resumen de estadísticas
tabla_estadisticas_ap = pd.DataFrame({
    "estadístico": ["Cantidad", "Media", "Mediana", "Mínimo", "Máximo", "Q1 (25%)", "Q3 (75%)"],
    "valor (USD)": [
        len(precios_ap),
        f"{precios_ap.mean():,.2f}",
        f"{precios_ap.median():,.2f}",
        f"{precios_ap.min():,.2f}",
        f"{precios_ap.max():,.2f}",
        f"{precios_ap.quantile(0.25):,.2f}",
        f"{precios_ap.quantile(0.75):,.2f}",
    ],
})

print(tabla_estadisticas_ap.to_string(index=False))

### Histograma de precios

El histograma muestra cuántos anuncios hay en cada rango de precios, permitiendo observar la distribución y detectar concentraciones de precios.

In [ ]:
# Histograma de precios de apartamentos
plt.figure(figsize=(8, 5))
plt.hist(precios_ap, bins=20, edgecolor="black", color="steelblue")
plt.title("Distribución de precios de apartamentos en USD")
plt.xlabel("Precio (USD)")
plt.ylabel("Número de anuncios")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Diagrama de caja (boxplot)

El boxplot muestra la dispersión de los precios, la mediana, los cuartiles y los posibles valores atípicos individuales.

In [ ]:
# Diagrama de caja (boxplot) de apartamentos
plt.figure(figsize=(6, 5))
plt.boxplot(
    precios_ap,
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", edgecolor="black"),
    medianprops=dict(color="red", linewidth=2),
)
plt.title("Boxplot de precios de apartamentos en USD")
plt.ylabel("Precio (USD)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación

Las estadísticas y los gráficos permiten observar cómo se distribuyen los precios de los apartamentos en USD en La Habana. La diferencia entre la media y la mediana puede indicar una distribución asimétrica: si la media es mayor que la mediana, sugiere que hay algunos precios muy elevados que tiran del promedio hacia arriba, mientras que la mayoría de los anuncios se concentran en valores más bajos.

## 10. Distribución de precios por municipio

Antes de comparar precios entre municipios, evaluamos si los datos permiten realizar este análisis de manera razonable. Revisamos cuántos anuncios tienen municipio identificado, cómo se distribuyen por municipio y cuántos anuncios tienen precio en USD por municipio.

In [ ]:
# Disponibilidad de municipios
total = len(df)
con_mun = df[df["municipio"].notna()]
sin_mun = df[df["municipio"].isna()]

print(f"Total de anuncios: {total}")
print(f"Con municipio identificado: {len(con_mun)}")
print(f"Sin municipio: {len(sin_mun)}")
print(f"Porcentaje con municipio: {len(con_mun)/total*100:.2f}%")
print(f"Porcentaje sin municipio: {len(sin_mun)/total*100:.2f}%")

In [ ]:
# Distribución por municipio
municipios_df = df[df["municipio"].notna()].copy()

dist = municipios_df.groupby("municipio").agg(
    cantidad=("id_fuente", "count"),
    con_precio=("precio", lambda x: x.notna().sum()),
    con_precio_usd=("precio", lambda x: ((x.notna()) & (df.loc[x.index, "moneda"] == "USD")).sum())
).sort_values("cantidad", ascending=False)
dist["porcentaje_total"] = (dist["cantidad"] / total * 100).round(2)
dist = dist.reset_index()

print(dist.to_string(index=False))

In [ ]:
# Precios en USD por municipio
df_usd = df[(df["municipio"].notna()) & (df["precio"].notna()) & (df["moneda"] == "USD")].copy()

precios_por_mun = df_usd.groupby("municipio").agg(
    cantidad=("precio", "count"),
    media=("precio", "mean"),
    mediana=("precio", "median")
).sort_values("cantidad", ascending=False)
precios_por_mun["media"] = precios_por_mun["media"].round(2)
precios_por_mun["mediana"] = precios_por_mun["mediana"].round(2)

print(precios_por_mun.to_string())

### Valoración metodológica

- **¿Qué proporción del conjunto tiene municipio identificado?** Alrededor del 61 % de los 307 anuncios tienen municipio asignado; el 39 % restante no lo tiene.
- **¿Hay municipios con muy pocos anuncios?** Sí. Cotorro tiene 4 anuncios y Regla tiene 3, los más bajos del listado.
- **¿La cobertura municipal depende principalmente de GAOS o de Revolico?** Depende principalmente de GAOS: identifica municipio en el 81.5 % de sus anuncios, mientras que Revolico solo lo hace en el 22.4 %.
- **¿Qué limitaciones debemos tener presentes antes de comparar precios entre municipios?** Más del 30 % de los anuncios no tienen municipio, por lo que cualquier comparación está sesgada hacia los anuncios que sí lo tienen. Además, algunos municipios tienen muy pocos anuncios (menos de 5), lo que hace que sus promedios y medianas no sean representativos. Finalmente, no todos los anuncios con municipio tienen precio en USD, lo que reduce aún más la muestra usable para comparaciones monetarias.

**Conclusión metodológica:** Debido a la escasez de anuncios con precio en USD en muchos municipios y al desequilibrio en la cobertura de fuentes, cualquier comparación de precios entre municipios debe interpretarse con cautela y solo para los municipios con suficiente cantidad de anuncios.

## 11. Composición municipal por tipo de inmueble

Examinamos cómo se distribuyen los tipos de inmueble dentro de cada municipio. Esto nos permite interpretar mejor las diferencias de precios observadas en la sección anterior, ya que un municipio con predominio de casas no es directamente comparable a uno con predominio de apartamentos.

In [ ]:
# Distribución por tipo de inmueble y municipio
municipios_df = df[df["municipio"].notna()].copy()

def clasificar_tipo(tipo):
    if pd.isna(tipo):
        return "Sin tipo"
    elif tipo == "casa":
        return "Casa"
    elif tipo == "apartamento":
        return "Apartamento"
    else:
        return "Otro/Indeterminado"

municipios_df["tipo_cat"] = municipios_df["tipo_inmueble"].apply(clasificar_tipo)

tabla_tipo = municipios_df.groupby(["municipio", "tipo_cat"]).size().unstack(fill_value=0)
tabla_tipo["Total"] = tabla_tipo.sum(axis=1)
tabla_tipo = tabla_tipo.sort_values("Total", ascending=False)

print(tabla_tipo.to_string())

### Valoración metodológica

- **¿Los municipios tienen una composición similar de casas y apartamentos?** No. Algunos municipios tienen predominio de casas (por ejemplo, Playa con 17 casas frente a 7 apartamentos; Boyeros con 9 casas frente a 2 apartamentos), mientras que otros tienen más apartamentos (Plaza de la Revolución con 22 apartamentos frente a 12 casas; Centro Habana con 12 apartamentos frente a 4 casas).
- **¿Qué implica esto para la comparación de precios?** Que los precios de casas y apartamentos no son directamente comparables entre municipios si la composición de tipos de inmueble es muy diferente.
- **Limitación adicional:** Muchos municipios tienen muy pocos anuncios cuando se separan por tipo de inmueble (por ejemplo, solo 1 apartamento en San Miguel del Padrón o Cotorro), lo que hace que sus medianas no sean representativas.


## 12. Características de las viviendas por precio relativo a la mediana (USD)

En esta sección respondemos a la pregunta: ¿qué características aparecen con mayor frecuencia en los anuncios cuyo precio supera la mediana? Para ello, filtramos los anuncios que cumplen tres condiciones: sean clasificados como `casa` o `apartamento`, tengan precio registrado y estén expresados en dólares estadounidenses (USD). No convertimos monedas ni mezclamos con EUR, y no eliminamos precios extremos para conservar la información original.

Separamos los anuncios de casas y de apartamentos en dos grupos: aquellos con precio superior a la mediana y aquellos con precio igual o inferior. Luego comparamos la distribución de habitaciones y baños entre ambos grupos.

**Nota:** Los porcentajes se calculan solo sobre los anuncios que tienen dato para cada característica, no sobre el total del grupo.

In [ ]:
# Filtrar anuncios de casas y apartamentos en USD
df_casas_usd = df[
    (df["tipo_inmueble"] == "casa") &
    (df["precio"].notna()) &
    (df["moneda"] == "USD")
].copy()

df_apartamentos_usd = df[
    (df["tipo_inmueble"] == "apartamento") &
    (df["precio"].notna()) &
    (df["moneda"] == "USD")
].copy()

# Calcular medianas por tipo de inmueble
mediana_casas = df_casas_usd["precio"].median()
mediana_apartamentos = df_apartamentos_usd["precio"].median()

print(f'Casas en USD: {len(df_casas_usd)}')
print(f'Mediana de casas: {mediana_casas:,.2f} USD')
print()
print(f'Apartamentos en USD: {len(df_apartamentos_usd)}')
print(f'Mediana de apartamentos: {mediana_apartamentos:,.2f} USD')


In [ ]:
# Separar en grupos por encima e igual o inferior a la mediana
casas_arriba = df_casas_usd[df_casas_usd["precio"] > mediana_casas].copy()
casas_abajo = df_casas_usd[df_casas_usd["precio"] <= mediana_casas].copy()

apartamentos_arriba = df_apartamentos_usd[df_apartamentos_usd["precio"] > mediana_apartamentos].copy()
apartamentos_abajo = df_apartamentos_usd[df_apartamentos_usd["precio"] <= mediana_apartamentos].copy()

print(f'Casas arriba de la mediana: {len(casas_arriba)}')
print(f'Casas igual o inferior a la mediana: {len(casas_abajo)}')
print()
print(f'Apartamentos arriba de la mediana: {len(apartamentos_arriba)}')
print(f'Apartamentos igual o inferior a la mediana: {len(apartamentos_abajo)}')


In [ ]:
# Tablas de distribución de habitaciones y baños por grupo

def tabla_distribucion(grupo, nombre_grupo):
    print(f'\n--- {nombre_grupo} ---')
    print(f'Total anuncios: {len(grupo)}')
    print(f'Con dato de habitaciones: {grupo["habitaciones"].notna().sum()}')
    print(f'Sin dato de habitaciones: {grupo["habitaciones"].isna().sum()}')
    print(f'Con dato de baños: {grupo["banos"].notna().sum()}')
    print(f'Sin dato de baños: {grupo["banos"].isna().sum()}')
    print()
    print('Distribución de habitaciones:')
    hab = grupo['habitaciones'].value_counts().sort_index()
    n_hab = grupo['habitaciones'].notna().sum()
    for v, c in hab.items():
        print(f'  {int(v)}: {c} ({c/n_hab*100:.1f}%)')
    print()
    print('Distribución de baños:')
    ban = grupo['banos'].value_counts().sort_index()
    n_ban = grupo['banos'].notna().sum()
    for v, c in ban.items():
        print(f'  {int(v)}: {c} ({c/n_ban*100:.1f}%)')
    print()

tabla_distribucion(casas_arriba, 'CASAS ARRIBA DE LA MEDIANA')
tabla_distribucion(casas_abajo, 'CASAS ABAJO O IGUAL A LA MEDIANA')
tabla_distribucion(apartamentos_arriba, 'APARTAMENTOS ARRIBA DE LA MEDIANA')
tabla_distribucion(apartamentos_abajo, 'APARTAMENTOS ABAJO O IGUAL A LA MEDIANA')

### Interpretación

Las tablas anteriores muestran cómo se distribuyen las características de habitaciones y baños entre los anuncios cuyos precios están por encima y por debajo (o igual) a la mediana, separados por tipo de inmueble y expresados en USD.

En las **casas**, los anuncios por encima de la mediana tienden a tener más habitaciones y más baños que los anuncios por debajo de la mediana. Sin embargo, una proporción considerable de anuncios (alrededor del 35 %) no informa el número de habitaciones ni de baños, por lo que estos resultados deben interpretarse con cautela.

En los **apartamentos**, también se observa que los anuncios por encima de la mediana tienden a tener más habitaciones y baños que los anuncios por debajo de la mediana, aunque una proporción considerable de anuncios (alrededor del 25-35 %) no informa estas características.

Estos resultados son descriptivos y no implican una relación de causalidad: tener más habitaciones o baños no implica necesariamente un precio más alto, ya que otros factores no disponibles en los datos (ubicación exacta, estado de conservación, acabados, etc.) también influyen en el precio.

Además, varias categorías tienen muy pocos anuncios (por ejemplo, casas con 7, 8 o 10 habitaciones, o apartamentos con 4 o 5 habitaciones), por lo que no se deben sacar conclusiones firmes basadas en esos casos aislados.

## 13. Inmueble representativo y comparación con el salario mínimo legal

En esta sección respondemos a la pregunta: ¿cuál es el inmueble más representativo de los anuncios analizados y cómo se compara su precio con el salario mínimo legal?

Se analizan casas y apartamentos por separado, utilizando los anuncios con precio en USD (la misma población de las preguntas anteriores). No se combina información de casas y apartamentos.


### 13.1. Características más frecuentes de los inmuebles

Se identifican, por separado para casas y apartamentos: el tipo de inmueble más frecuente, la cantidad de habitaciones más frecuente, la cantidad de baños más frecuente y la mediana del precio.

**Nota:** Las frecuencias de habitaciones y baños se calculan solo sobre los anuncios que informan ese dato. Se presentan como modas individuales por característica; no se afirma que exista una combinación de habitaciones y baños que represente a la mayoría de los inmuebles, ya que los datos no permiten identificar una combinación simultánea mayoritaria.


In [ ]:
import json
from collections import Counter

with open('data/processed/anuncios/unificado_procesado_83924fda_20261115_034632.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)

anuncios = datos['datos']

# Filtrar casas y apartamentos con precio en USD (misma población de las preguntas anteriores)
casas_usd = [
    a for a in anuncios
    if a.get('tipo_inmueble') == 'casa'
    and isinstance(a.get('precio'), (int, float))
    and a.get('precio') is not None
    and a.get('moneda') == 'USD'
]

apartamentos_usd = [
    a for a in anuncios
    if a.get('tipo_inmueble') == 'apartamento'
    and isinstance(a.get('precio'), (int, float))
    and a.get('precio') is not None
    and a.get('moneda') == 'USD'
]

def mediana(vals):
    s = sorted(vals)
    n = len(s)
    if n == 0:
        return None
    if n % 2 == 1:
        return s[n // 2]
    return (s[n // 2 - 1] + s[n // 2]) / 2

def moda_con_dato(lista, clave):
    """Devuelve (valor_moda, conteo_moda, n_con_dato, n_total, n_sin_dato)."""
    c = Counter()
    n_disp = 0
    for a in lista:
        x = a.get(clave)
        if x is not None:
            c[x] += 1
            n_disp += 1
    if not c:
        return None, 0, 0, len(lista), len(lista)
    modas = sorted(c.items(), key=lambda item: (-item[1], item[0]))
    return modas[0][0], modas[0][1], n_disp, len(lista), len(lista) - n_disp

# Casas
precios_casas = [a['precio'] for a in casas_usd]
mediana_casas = mediana(precios_casas)
hab_casa, n_hab_casa, disp_hab_casa, tot_casa, sin_hab_casa = moda_con_dato(casas_usd, 'habitaciones')
ban_casa, n_ban_casa, disp_ban_casa, _, sin_ban_casa = moda_con_dato(casas_usd, 'banos')

# Apartamentos
precios_ap = [a['precio'] for a in apartamentos_usd]
mediana_ap = mediana(precios_ap)
hab_ap, n_hab_ap, disp_hab_ap, tot_ap, sin_hab_ap = moda_con_dato(apartamentos_usd, 'habitaciones')
ban_ap, n_ban_ap, disp_ban_ap, _, sin_ban_ap = moda_con_dato(apartamentos_usd, 'banos')

print('=== CASAS (en USD) ===')
print(f'Anuncios: {tot_casa}')
print(f'Mediana de precio: {mediana_casas:,.2f} USD')
print(f'Habitaciones - moda: {hab_casa}, anuncios: {n_hab_casa}, con dato: {disp_hab_casa}, sin dato: {sin_hab_casa}, porcentaje: {n_hab_casa/disp_hab_casa*100:.1f}%')
print(f'Banos - moda: {ban_casa}, anuncios: {n_ban_casa}, con dato: {disp_ban_casa}, sin dato: {sin_ban_casa}, porcentaje: {n_ban_casa/disp_ban_casa*100:.1f}%')
print()
print('=== APARTAMENTOS (en USD) ===')
print(f'Anuncios: {tot_ap}')
print(f'Mediana de precio: {mediana_ap:,.2f} USD')
print(f'Habitaciones - moda: {hab_ap}, anuncios: {n_hab_ap}, con dato: {disp_hab_ap}, sin dato: {sin_hab_ap}, porcentaje: {n_hab_ap/disp_hab_ap*100:.1f}%')
print(f'Banos - moda: {ban_ap}, anuncios: {n_ban_ap}, con dato: {disp_ban_ap}, sin dato: {sin_ban_ap}, porcentaje: {n_ban_ap/disp_ban_ap*100:.1f}%')


### 13.2. Comparación con el salario mínimo legal

Se compara la mediana del precio de las casas (30.000 USD) con el salario mínimo legal mensual de 3.210 CUP. Se presentan dos escenarios, utilizando tasas de cambio del 15 de noviembre de 2026.

**Fórmula:** `(30.000 × tasa USD/CUP) / 3.210`

El resultado se expresa en meses y años de salario íntegro, suponiendo hipotéticamente que se ahorrara el 100 % del salario. No es una estimación del tiempo real necesario para comprar una vivienda.

**Fuentes:**
- Salario mínimo legal: dato normativo externo (3.210 CUP mensuales).
- elTOQUE: tasa del historial local (`data/raw/exchange_rates/historial_tasas.json`), fecha 15/11/2026.
- BCC: tasa obtenida de la API oficial (`api.bc.gob.cu/v1/tasas-de-cambio/activas`), fecha 15/11/2026. La tasa `tasaEspecial` se identifica como correspondiente al Segmento III («Tasa Población y FGNE») a partir de la información oficial consultada; la respuesta JSON de la API no incluye una etiqueta explícita de segmento.

La tasa del BCC no está incorporada al historial local de tasas; se obtuvo mediante consulta directa a la API.


In [ ]:
mediana_casas_usd = 30000
salario_minimo_cup = 3210

# Tasas de cambio (15/11/2026)
tasa_eltoque = 920.0      # elTOQUE, mercado informal, historial local
tasa_bcc = 832.0          # BCC, Segmento III (tasaEspecial), API oficial

print('=== COMPARACIÓN CON SALARIO MÍNIMO LEGAL ===')
print(f'Mediana de casas: {mediana_casas_usd:,.2f} USD')
print(f'Salario mínimo mensual: {salario_minimo_cup} CUP')
print()

# Escenario elTOQUE
precio_cup_eltoque = mediana_casas_usd * tasa_eltoque
meses_eltoque = precio_cup_eltoque / salario_minimo_cup
anios_eltoque = meses_eltoque / 12
print('--- Escenario elTOQUE (mercado informal) ---')
print(f'Tasa: {tasa_eltoque} CUP/USD (15/11/2026)')
print(f'Equivalencia en CUP: {precio_cup_eltoque:,.0f} CUP')
print(f'Meses de salario íntegro: {meses_eltoque:,.1f}')
print(f'Años de salario íntegro: {anios_eltoque:,.1f}')
print()

# Escenario BCC
precio_cup_bcc = mediana_casas_usd * tasa_bcc
meses_bcc = precio_cup_bcc / salario_minimo_cup
anios_bcc = meses_bcc / 12
print('--- Escenario BCC (tasa oficial, Segmento III) ---')
print(f'Tasa: {tasa_bcc} CUP/USD (15/11/2026)')
print(f'Equivalencia en CUP: {precio_cup_bcc:,.0f} CUP')
print(f'Meses de salario íntegro: {meses_bcc:,.1f}')
print(f'Años de salario íntegro: {anios_bcc:,.1f}')


### Interpretación

En las **casas**, la cantidad de habitaciones más frecuente es 3 (31 anuncios, 30,1 % de los 103 que informan este dato) y la cantidad de baños más frecuente es 1 (43 anuncios, 43,4 % de los 99 que informan este dato). En los **apartamentos**, la cantidad de habitaciones más frecuente es 2 (41 anuncios, 46,6 % de los 88 que informan) y la cantidad de baños más frecuente es 1 (64 anuncios, 77,1 % de los 83 que informan).

Estas son frecuencias individuales por característica. No se puede afirmar que exista una combinación de habitaciones y baños que represente a la mayoría de los inmuebles, ya que los datos no permiten identificar una combinación simultánea mayoritaria.

Respecto al salario mínimo, la mediana de una casa (30.000 USD) equivale a aproximadamente 8.598 meses de salario íntegro con la tasa de elTOQUE y a 7.776 meses con la tasa oficial del BCC (Segmento III). En ambos casos se trata de una equivalencia hipotética que supone ahorrar el 100 % del salario durante varios años, no de una estimación del tiempo real necesario para comprar una vivienda.

### Limitaciones

- Entre el 25 % y el 50 % de los anuncios no informan habitaciones o baños, según el grupo; los porcentajes se calculan solo sobre los que tienen dato.
- Las frecuencias son individuales y no permiten afirmar una combinación típica conjunta de habitaciones y baños.
- El salario mínimo es un dato normativo externo, no proviene de los anuncios.
- La comparación salarial es hipotética (ahorro del 100 % del salario), no una estimación real.
- Las tasas de cambio corresponden al 15/11/2026 y no representan necesariamente la situación de otras fechas.
- La tasa del BCC no está incorporada al historial local; se obtuvo mediante consulta directa a la API oficial.
